# Delhi-NCR Weather & Aerosol Data Fusion Pipeline

This notebook ingests, parses, and fuses spatial weather observation data from three distinct regional NOAA meteorological stations alongside daily NASA AERONET Aerosol Optical Depth (AOD) column records. 

### 🏗️ Data Architecture & Offset Logic
Observations are parsed directly from the mandatory fixed-width control sections (indices 0 to 104) of every row rather than searching for optional remarks, ensuring 100% of the raw observations are successfully recovered without gaps. All stations are resampled and normalized to a strict hourly temporal resolution matching the entire year of 2024 ($n=8784$ records).

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

def parse_noaa_isd(file_name, station_name):
    """
    Parses raw NOAA ISD fixed-width ASCII weather records using stable physical byte offsets.
    Extracts metrics from the mandatory 105-character observation prefix section.
    """
    project_root = Path(os.getcwd()).resolve()
    if project_root.name == "notebooks":
        project_root = project_root.parent
        
    file_path = project_root / "data" / "raw" / file_name
    print(f"Ingesting {station_name} dataset from: {file_path}")
    
    if not file_path.exists():
        raise FileNotFoundError(f"🚨 Raw weather file not found at: {file_path}")
        
    parsed_rows = []
    
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if len(line) < 105:
                continue
                
            try:
                # The 4-digit year '2024' always starts exactly at index 15 in NOAA ISD
                year = line[15:19]
                if year != "2024":
                    continue
                    
                month = line[19:21]
                day = line[21:23]
                hour = line[23:25]
                minute = line[25:27]
                
                timestamp = pd.to_datetime(f"{year}-{month}-{day} {hour}:{minute}:00")
                
                # Wind Features: Direction (60:63), Speed (65:69)
                wind_dir_raw = line[60:63]
                wind_dir = float(wind_dir_raw) if wind_dir_raw != "999" else np.nan
                
                wind_speed_raw = line[65:69]
                wind_speed = float(wind_speed_raw) / 10.0 if wind_speed_raw != "9999" else np.nan
                
                # Visibility Range (78:84)
                vis_raw = line[78:84]
                visibility = float(vis_raw) if vis_raw != "999999" else np.nan
                
                # Air Temperature (87:92)
                temp_raw = line[87:92]
                if temp_raw in ("+9999", "99999"):
                    temp = np.nan
                else:
                    temp = float(temp_raw[1:5]) / 10.0
                    if temp_raw[0] == '-':
                        temp = -temp
                        
                # Dew Point Temperature (93:98)
                dew_raw = line[93:98]
                if dew_raw in ("+9999", "99999"):
                    dew = np.nan
                else:
                    dew = float(dew_raw[1:5]) / 10.0
                    if dew_raw[0] == '-':
                        dew = -dew
                        
                # Sea Level Pressure (99:104)
                slp_raw = line[99:104]
                slp = float(slp_raw) / 10.0 if slp_raw != "99999" else np.nan
                
                parsed_rows.append({
                    'timestamp': timestamp,
                    f'{station_name}_temp': temp,
                    f'{station_name}_dew': dew,
                    f'{station_name}_wind_speed': wind_speed,
                    f'{station_name}_wind_dir': wind_dir,
                    f'{station_name}_slp': slp,
                    f'{station_name}_visibility': visibility
                })
            except Exception:
                continue
                
    df = pd.DataFrame(parsed_rows)
    print(f"  Parsed {len(df)} records.")
    
    # Aggregate into strict hourly windows and reindex onto the complete 2024 calendar
    df = df.set_index('timestamp').resample('1h').mean()
    time_index = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
    df = df.reindex(time_index)
    df.index.name = 'timestamp'
    df = df.reset_index()
    
    return df

In [11]:
def fuse_aerosols_and_export(df_air, df_urb, df_rur, aeronet_file, output_file):
    """
    Standardizes daily NASA AERONET AOD data and left-joins it onto the uniform hourly meteorological matrix.
    """
    project_root = Path(os.getcwd()).resolve()
    if project_root.name == "notebooks":
        project_root = project_root.parent
        
    aeronet_path = project_root / "data" / "raw" / aeronet_file
    output_path = project_root / "data" / "processed" / output_file
    
    print(f"\nLoading NASA Aerosol AOD data from: {aeronet_path}")
    df_aero = pd.read_csv(aeronet_path)
    df_aero.columns = df_aero.columns.str.strip()
    
    df_aero['date_parsed'] = pd.to_datetime(df_aero['Date(dd:mm:yyyy)'], format='%d:%m:%Y')
    
    target_cols = ['AOD_500nm', 'AOD_440nm', 'AOD_675nm', '440-870_Angstrom_Exponent']
    for col in target_cols:
        if col in df_aero.columns:
            df_aero[col] = df_aero[col].replace(-999.0, np.nan)
            
    df_daily_aero = df_aero.groupby('date_parsed')[target_cols].mean().reset_index()
    df_daily_aero = df_daily_aero.rename(columns={'date_parsed': 'date_key'})
    
    print("Fusing meteorological datasets...")
    master_df = pd.merge(df_air, df_urb, on='timestamp', how='outer')
    master_df = pd.merge(master_df, df_rur, on='timestamp', how='outer')
    
    master_df['date_key'] = master_df['timestamp'].dt.normalize()
    
    print("Merging observations with atmospheric aerosol indexes...")
    final_fused_df = pd.merge(master_df, df_daily_aero, on='date_key', how='left')
    final_fused_df = final_fused_df.drop(columns=['date_key'])
    final_fused_df = final_fused_df.sort_values('timestamp').reset_index(drop=True)
    
    os.makedirs(output_path.parent, exist_ok=True)
    final_fused_df.to_csv(output_path, index=False)
    
    print(f"\n🎯 SUCCESS: Unified spatial matrix created and saved to:\n 💾 {output_path}")
    print(f"Total structured dimensions: {final_fused_df.shape}")
    return final_fused_df

In [12]:
print("=== STARTING CLIMATE VISIBILITY DATA FUSION PIPELINE ===\n")

# 1. Parse weather stations cleanly using robust fixed-width slice logic
df_airport = parse_noaa_isd("421810-99999-2024.txt", "airport")
df_safdarjung = parse_noaa_isd("421820-99999-2024.txt", "urban")
df_rohtak = parse_noaa_isd("421390-99999-2024.txt", "rural")

# 2. Run the daily aerosol fusion layer and save output
df_master = fuse_aerosols_and_export(df_airport, df_safdarjung, df_rohtak, 
                                     "aeronet_aerosols_raw.csv", 
                                     "delhi_2024_master_fused.csv")

=== STARTING CLIMATE VISIBILITY DATA FUSION PIPELINE ===

Ingesting airport dataset from: /Users/vedikaagrawal/Documents/climate-visibility-new/data/raw/421810-99999-2024.txt
  Parsed 19511 records.
Ingesting urban dataset from: /Users/vedikaagrawal/Documents/climate-visibility-new/data/raw/421820-99999-2024.txt
  Parsed 2888 records.
Ingesting rural dataset from: /Users/vedikaagrawal/Documents/climate-visibility-new/data/raw/421390-99999-2024.txt
  Parsed 1392 records.

Loading NASA Aerosol AOD data from: /Users/vedikaagrawal/Documents/climate-visibility-new/data/raw/aeronet_aerosols_raw.csv
Fusing meteorological datasets...
Merging observations with atmospheric aerosol indexes...

🎯 SUCCESS: Unified spatial matrix created and saved to:
 💾 /Users/vedikaagrawal/Documents/climate-visibility-new/data/processed/delhi_2024_master_fused.csv
Total structured dimensions: (8784, 23)


In [13]:
print("\n=== MASTER MATRIX PREVIEW ===")
df_master.head()


=== MASTER MATRIX PREVIEW ===


,timestamp,airport_temp,airport_dew,airport_wind_speed,airport_wind_dir,airport_slp,airport_visibility,urban_temp,urban_dew,urban_wind_speed,...,rural_temp,rural_dew,rural_wind_speed,rural_wind_dir,rural_slp,rural_visibility,AOD_500nm,AOD_440nm,AOD_675nm,440-870_Angstrom_Exponent
0,2024-01-01 00:00:00,12.066667,9.666667,2.10,273.333333,1019.4,966.666667,10.8,10.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01 01:00:00,11.000000,9.000000,0.75,330.000000,NaN,1150.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-01 02:00:00,11.000000,9.000000,2.10,280.000000,NaN,1200.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-01 03:00:00,11.066667,8.800000,1.50,223.333333,1021.2,1133.333333,NaN,NaN,NaN,...,10.4,7.9,0.0,NaN,1020.5,2000.0,NaN,NaN,NaN,NaN
4,2024-01-01 04:00:00,11.000000,9.000000,1.50,30.000000,NaN,1250.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
print("=== MATRIX COLUMN NULL PERCENTAGES ===")
null_pct = df_master.isnull().mean() * 100
for col in df_master.columns:
    print(f"{col:<30}: {null_pct[col]:.2f}%")

=== MATRIX COLUMN NULL PERCENTAGES ===
timestamp                     : 0.00%
airport_temp                  : 4.17%
airport_dew                   : 4.17%
airport_wind_speed            : 4.17%
airport_wind_dir              : 18.70%
airport_slp                   : 68.60%
airport_visibility            : 4.17%
urban_temp                    : 67.86%
urban_dew                     : 67.86%
urban_wind_speed              : 67.85%
urban_wind_dir                : 83.72%
urban_slp                     : 68.62%
urban_visibility              : 67.85%
rural_temp                    : 84.15%
rural_dew                     : 84.15%
rural_wind_speed              : 84.15%
rural_wind_dir                : 91.14%
rural_slp                     : 84.15%
rural_visibility              : 84.15%
AOD_500nm                     : 34.97%
AOD_440nm                     : 34.97%
AOD_675nm                     : 34.43%
440-870_Angstrom_Exponent     : 34.43%
